Dataset download

In [2]:
!git clone https://github.com/laxmimerit/dog-cat-full-dataset.git


Cloning into 'dog-cat-full-dataset'...
remote: Enumerating objects: 25033, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 25033 (delta 0), reused 4 (delta 0), pack-reused 25027 (from 1)
Receiving objects: 100% (25033/25033), 541.85 MiB | 23.69 MiB/s, done.
Resolving deltas: 100% (5/5), done.
Updating files: 100% (24990/24990), done.


Loading data

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

def load_images_from_folder(folder, image_size=64, limit_per_class=None):
    data = []
    labels = []
    for label_name in ["cats", "dogs"]:
        path = os.path.join(folder, label_name)
        label = 0 if label_name == "cat" else 1
        files = os.listdir(path)
        if limit_per_class:
            files = files[:limit_per_class]
        for img_file in tqdm(files, desc=f"Loading {label_name}s"):
            img_path = os.path.join(path, img_file)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                img = cv2.resize(img, (image_size, image_size))
                data.append(img.flatten())
                labels.append(label)
    return np.array(data), np.array(labels)

# Load train data
X_train, y_train = load_images_from_folder("data/train", image_size=64, limit_per_class=1000)

# Load test data
X_test, y_test = load_images_from_folder("data/test", image_size=64, limit_per_class=500)


Loading dogss: 100%|██████████| 500/500 [00:00<00:00, 873.28it/s]


In [ ]:
print("Cats in train:", len(os.listdir("data/train/cats")))
print("Dogs in train:", len(os.listdir("data/train/dogs")))


Cats in train: 10000
Dogs in train: 9989


training process

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import shuffle

# Step 4: Load images using correct folder names (cats, dogs)
def load_images_from_folder(folder, image_size=64, limit_per_class=None):
    data = []
    labels = []
    class_counts = {"cats": 0, "dogs": 0}

    for label_name in ["cats", "dogs"]:
        path = os.path.join(folder, label_name)
        label = 0 if label_name == "cats" else 1

        if not os.path.exists(path):
            print(f"❌ Folder not found: {path}")
            continue

        files = sorted([
            f for f in os.listdir(path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png')) and os.path.isfile(os.path.join(path, f))
        ])

        if limit_per_class:
            files = files[:limit_per_class]

        if not files:
            print(f"⚠️ No images found in {path}")
            continue

        for img_file in tqdm(files, desc=f"Loading {label_name}"):
            img_path = os.path.join(path, img_file)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f"⚠️ Failed to read: {img_path}")
                continue
            img = cv2.resize(img, (image_size, image_size))
            data.append(img.flatten())
            labels.append(label)
            class_counts[label_name] += 1

    if not data:
        raise ValueError("❌ No images were loaded. Please check dataset paths and formats.")

    print(f"\n✅ Loaded {class_counts['cats']} cats and {class_counts['dogs']} dogs.")
    return np.array(data), np.array(labels)

# Step 5: Load dataset (e.g., 1000 per class for speed)
X_train, y_train = load_images_from_folder("data/train", image_size=64, limit_per_class=1000)
X_test, y_test = load_images_from_folder("data/test", image_size=64, limit_per_class=500)

# Step 6: Normalize and shuffle
X_train = X_train / 255.0
X_test = X_test / 255.0
X_train, y_train = shuffle(X_train, y_train, random_state=42)
X_test, y_test = shuffle(X_test, y_test, random_state=42)

# Step 7: Train SVM
print("\n🔧 Training SVM...")
clf = SVC(kernel='linear', C=1.0)
clf.fit(X_train, y_train)

# Step 8: Evaluate
y_pred = clf.predict(X_test)
print("\n📊 Accuracy:", accuracy_score(y_test, y_pred))
print("\n📝 Classification Report:\n", classification_report(y_test, y_pred, target_names=["Cat", "Dog"]))


Loading dogs: 100%|██████████| 9989/9989 [00:08<00:00, 1239.11it/s]



✅ Loaded 10000 cats and 9989 dogs.


Loading dogs: 100%|██████████| 500/500 [00:00<00:00, 1418.34it/s]



✅ Loaded 500 cats and 500 dogs.

🔧 Training SVM...


In [ ]:
!pip install -q opencv-python scikit-image tqdm scikit-learn

❌ Folder not found: data/train/cat
❌ Folder not found: data/train/dog

✅ Loaded 0 cats and 0 dogs.
Shape of X_train: (0,)
Sample labels: (array([], dtype=float64), array([], dtype=int64))


feature extract

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import shuffle
from sklearn.cluster import MiniBatchKMeans

# -- Step 1: Load all grayscale images from folder, no limit
def load_images(folder, image_size=64):
    images = []
    labels = []
    for label_name in ["cats", "dogs"]:
        path = os.path.join(folder, label_name)
        label = 0 if label_name == "cats" else 1
        if not os.path.exists(path):
            print(f"Folder not found: {path}")
            continue
        files = sorted([f for f in os.listdir(path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        for f in tqdm(files, desc=f"Loading {label_name} images"):
            img_path = os.path.join(path, f)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img = cv2.resize(img, (image_size, image_size))
            images.append(img)
            labels.append(label)
    return np.array(images), np.array(labels)

# -- Step 2: Extract HOG features for each image
def extract_hog_features(images, pixels_per_cell=(8,8), cells_per_block=(2,2)):
    hog_features = []
    for img in images:
        fd = hog(img, orientations=9, pixels_per_cell=pixels_per_cell,
                 cells_per_block=cells_per_block, block_norm='L2-Hys')
        hog_features.append(fd)
    return np.array(hog_features)

# -- Step 3: Extract ORB descriptors for all images
def extract_orb_descriptors(images):
    orb = cv2.ORB_create(nfeatures=500)
    all_descriptors = []
    img_descriptors = []  # to keep descriptors per image
    for img in tqdm(images, desc="Extracting ORB descriptors"):
        keypoints, des = orb.detectAndCompute(img, None)
        if des is None:
            des = np.array([], dtype=np.float32).reshape(0, 32)  # empty descriptor
        img_descriptors.append(des)
        if des.shape[0] > 0:
            all_descriptors.append(des)
    if all_descriptors:
        all_descriptors = np.vstack(all_descriptors)
    else:
        all_descriptors = np.array([], dtype=np.float32).reshape(0, 32)
    return img_descriptors, all_descriptors

# -- Step 4: Create a visual vocabulary (codebook) by clustering ORB descriptors with KMeans
def create_codebook(descriptors, k=50):
    print("Clustering descriptors with MiniBatchKMeans...")
    kmeans = MiniBatchKMeans(n_clusters=k, batch_size=1000, random_state=42)
    kmeans.fit(descriptors)
    return kmeans

# -- Step 5: Compute ORB histogram (BoW) for each image descriptors
def compute_bow_histograms(kmeans, img_descriptors, k=50):
    histograms = []
    for des in tqdm(img_descriptors, desc="Computing BoW histograms"):
        if des.shape[0] == 0:
            hist = np.zeros(k)
        else:
            labels = kmeans.predict(des)
            hist, _ = np.histogram(labels, bins=np.arange(k+1))
        histograms.append(hist)
    return np.array(histograms)

# -- Load train and test images
X_train_imgs, y_train = load_images("data/train")
X_test_imgs, y_test = load_images("data/test")

# Extract HOG features
X_train_hog = extract_hog_features(X_train_imgs)
X_test_hog = extract_hog_features(X_test_imgs)

# Extract ORB descriptors
train_img_des, train_all_des = extract_orb_descriptors(X_train_imgs)
test_img_des, test_all_des = extract_orb_descriptors(X_test_imgs)

# Build codebook on training ORB descriptors
k = 50  # number of visual words
if train_all_des.shape[0] == 0:
    print("No ORB descriptors found in training images. Skipping ORB features.")
    X_train_orb_bow = np.zeros((len(X_train_imgs), k))
    X_test_orb_bow = np.zeros((len(X_test_imgs), k))
else:
    kmeans = create_codebook(train_all_des, k=k)
    X_train_orb_bow = compute_bow_histograms(kmeans, train_img_des, k=k)
    X_test_orb_bow = compute_bow_histograms(kmeans, test_img_des, k=k)

# Concatenate HOG + ORB-BoW features
X_train_feat = np.hstack([X_train_hog, X_train_orb_bow])
X_test_feat = np.hstack([X_test_hog, X_test_orb_bow])

# Normalize features
scaler = StandardScaler()
X_train_feat = scaler.fit_transform(X_train_feat)
X_test_feat = scaler.transform(X_test_feat)

# Shuffle train and test sets
X_train_feat, y_train = shuffle(X_train_feat, y_train, random_state=42)
X_test_feat, y_test = shuffle(X_test_feat, y_test, random_state=42)

# Train linear SVM
print("\nTraining SVM with combined HOG + ORB-BoW features...")
clf = SVC(kernel='linear', C=1.0)
clf.fit(X_train_feat, y_train)

# Evaluate
y_pred = clf.predict(X_test_feat)
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=["Cat", "Dog"]))

Extracting ORB descriptors: 100%|██████████| 5000/5000 [00:02<00:00, 2210.05it/s]


Clustering descriptors with MiniBatchKMeans...


Computing BoW histograms: 100%|██████████| 5000/5000 [00:00<00:00, 17610.21it/s]



Training SVM with combined HOG + ORB-BoW features...


In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import shuffle
from sklearn.cluster import MiniBatchKMeans
from concurrent.futures import ThreadPoolExecutor

# -- Step 1: Load all grayscale images from folder, downsized to 32x32 for speed

def load_images(folder, image_size=32):
    images = []
    labels = []
    for label_name in ["cats", "dogs"]:
        path = os.path.join(folder, label_name)
        label = 0 if label_name == "cats" else 1
        if not os.path.exists(path):
            print(f"Folder not found: {path}")
            continue
        files = sorted([f for f in os.listdir(path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        for f in tqdm(files, desc=f"Loading {label_name} images"):
            img_path = os.path.join(path, f)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img = cv2.resize(img, (image_size, image_size))
            images.append(img)
            labels.append(label)
    return np.array(images), np.array(labels)

# -- Step 2 & 3: Extract HOG + ORB in parallel

def extract_features_parallel(images, orb_features=200, image_size=32):
    orb = cv2.ORB_create(nfeatures=orb_features)

    def process(img):
        fd = hog(img, orientations=9, pixels_per_cell=(8,8),
                 cells_per_block=(2,2), block_norm='L2-Hys')
        keypoints, des = orb.detectAndCompute(img, None)
        if des is None:
            des = np.array([], dtype=np.float32).reshape(0, 32)
        return fd, des

    with ThreadPoolExecutor(max_workers=8) as executor:
        results = list(executor.map(process, images))

    hog_features = [r[0] for r in results]
    img_descriptors = [r[1] for r in results]
    all_descriptors = [r[1] for r in results if r[1].shape[0] > 0]

    if all_descriptors:
        all_descriptors = np.vstack(all_descriptors)
    else:
        all_descriptors = np.array([], dtype=np.float32).reshape(0, 32)

    return np.array(hog_features), img_descriptors, all_descriptors

# -- Step 4: Create visual vocabulary (codebook) using fewer clusters for speed

def create_codebook(descriptors, k=30):
    print("Clustering descriptors with MiniBatchKMeans...")
    kmeans = MiniBatchKMeans(n_clusters=k, batch_size=500, random_state=42)
    kmeans.fit(descriptors)
    return kmeans

# -- Step 5: Compute BoW histograms

def compute_bow_histograms(kmeans, img_descriptors, k=30):
    histograms = []
    for des in tqdm(img_descriptors, desc="Computing BoW histograms"):
        if des.shape[0] == 0:
            hist = np.zeros(k)
        else:
            labels = kmeans.predict(des)
            hist, _ = np.histogram(labels, bins=np.arange(k+1))
        histograms.append(hist)
    return np.array(histograms)

# -- Load train and test images
X_train_imgs, y_train = load_images("data/train", image_size=32)
X_test_imgs, y_test = load_images("data/test", image_size=32)

# Extract features in parallel
X_train_hog, train_img_des, train_all_des = extract_features_parallel(X_train_imgs, orb_features=200)
X_test_hog, test_img_des, test_all_des = extract_features_parallel(X_test_imgs, orb_features=200)

# Create visual codebook (BoW)
k = 30
if train_all_des.shape[0] == 0:
    print("No ORB descriptors found in training images. Skipping ORB features.")
    X_train_orb_bow = np.zeros((len(X_train_imgs), k))
    X_test_orb_bow = np.zeros((len(X_test_imgs), k))
else:
    kmeans = create_codebook(train_all_des, k=k)
    X_train_orb_bow = compute_bow_histograms(kmeans, train_img_des, k=k)
    X_test_orb_bow = compute_bow_histograms(kmeans, test_img_des, k=k)

# Combine features
X_train_feat = np.hstack([X_train_hog, X_train_orb_bow])
X_test_feat = np.hstack([X_test_hog, X_test_orb_bow])

# Normalize
scaler = StandardScaler()
X_train_feat = scaler.fit_transform(X_train_feat)
X_test_feat = scaler.transform(X_test_feat)

# Shuffle
X_train_feat, y_train = shuffle(X_train_feat, y_train, random_state=42)
X_test_feat, y_test = shuffle(X_test_feat, y_test, random_state=42)

# Train SVM
print("\nTraining SVM with combined HOG + ORB-BoW features...")
clf = SVC(kernel='linear', C=1.0)
clf.fit(X_train_feat, y_train)

# Evaluate
y_pred = clf.predict(X_test_feat)
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=["Cat", "Dog"]))


In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import shuffle
from sklearn.cluster import MiniBatchKMeans
from concurrent.futures import ThreadPoolExecutor

# -- Step 1: Load all grayscale images from folder, downsized to 32x32 for speed

def load_images(folder, image_size=32):
    images = []
    labels = []
    for label_name in ["cats", "dogs"]:
        path = os.path.join(folder, label_name)
        label = 0 if label_name == "cats" else 1
        if not os.path.exists(path):
            print(f"Folder not found: {path}")
            continue
        files = sorted([f for f in os.listdir(path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        for f in tqdm(files, desc=f"Loading {label_name} images"):
            img_path = os.path.join(path, f)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img = cv2.resize(img, (image_size, image_size))
            images.append(img)
            labels.append(label)
    return np.array(images), np.array(labels)

# -- Step 2 & 3: Extract HOG + ORB in parallel

def extract_features_parallel(images, orb_features=200, image_size=32):
    orb = cv2.ORB_create(nfeatures=orb_features)

    def process(img):
        fd = hog(img, orientations=9, pixels_per_cell=(8,8),
                 cells_per_block=(2,2), block_norm='L2-Hys')
        keypoints, des = orb.detectAndCompute(img, None)
        if des is None:
            des = np.array([], dtype=np.float32).reshape(0, 32)
        return fd, des

    with ThreadPoolExecutor(max_workers=8) as executor:
        results = list(executor.map(process, images))

    hog_features = [r[0] for r in results]
    img_descriptors = [r[1] for r in results]
    all_descriptors = [r[1] for r in results if r[1].shape[0] > 0]

    if all_descriptors:
        all_descriptors = np.vstack(all_descriptors)
    else:
        all_descriptors = np.array([], dtype=np.float32).reshape(0, 32)

    return np.array(hog_features), img_descriptors, all_descriptors

# -- Step 4: Create visual vocabulary (codebook) using fewer clusters for speed

def create_codebook(descriptors, k=30):
    print("Clustering descriptors with MiniBatchKMeans...")
    kmeans = MiniBatchKMeans(n_clusters=k, batch_size=500, random_state=42)
    kmeans.fit(descriptors)
    return kmeans

# -- Step 5: Compute BoW histograms

def compute_bow_histograms(kmeans, img_descriptors, k=30):
    histograms = []
    for des in tqdm(img_descriptors, desc="Computing BoW histograms"):
        if des.shape[0] == 0:
            hist = np.zeros(k)
        else:
            labels = kmeans.predict(des)
            hist, _ = np.histogram(labels, bins=np.arange(k+1))
        histograms.append(hist)
    return np.array(histograms)

# -- Load train and test images
X_train_imgs, y_train = load_images("data/train", image_size=32)
X_test_imgs, y_test = load_images("data/test", image_size=32)

# Extract features in parallel
X_train_hog, train_img_des, train_all_des = extract_features_parallel(X_train_imgs, orb_features=200)
X_test_hog, test_img_des, test_all_des = extract_features_parallel(X_test_imgs, orb_features=200)

# Create visual codebook (BoW)
k = 30
if train_all_des.shape[0] == 0:
    print("No ORB descriptors found in training images. Skipping ORB features.")
    X_train_orb_bow = np.zeros((len(X_train_imgs), k))
    X_test_orb_bow = np.zeros((len(X_test_imgs), k))
else:
    kmeans = create_codebook(train_all_des, k=k)
    X_train_orb_bow = compute_bow_histograms(kmeans, train_img_des, k=k)
    X_test_orb_bow = compute_bow_histograms(kmeans, test_img_des, k=k)

# Combine features
X_train_feat = np.hstack([X_train_hog, X_train_orb_bow])
X_test_feat = np.hstack([X_test_hog, X_test_orb_bow])

# Normalize
scaler = StandardScaler()
X_train_feat = scaler.fit_transform(X_train_feat)
X_test_feat = scaler.transform(X_test_feat)

# Shuffle
X_train_feat, y_train = shuffle(X_train_feat, y_train, random_state=42)
X_test_feat, y_test = shuffle(X_test_feat, y_test, random_state=42)

# Train SVM
print("\nTraining SVM with combined HOG + ORB-BoW features...")
clf = SVC(kernel='linear', C=1.0)
clf.fit(X_train_feat, y_train)

# Evaluate
y_pred = clf.predict(X_test_feat)
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=["Cat", "Dog"]))


Loading dogs images: 100%|██████████| 2500/2500 [00:01<00:00, 1398.48it/s]


No ORB descriptors found in training images. Skipping ORB features.

Training SVM with combined HOG + ORB-BoW features...


In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import hog
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import shuffle
from sklearn.cluster import MiniBatchKMeans
from concurrent.futures import ThreadPoolExecutor

# -- Step 1: Load all grayscale images from folder, downsized to 32x32 for speed

def load_images(folder, image_size=32):
    images = []
    labels = []
    for label_name in ["cats", "dogs"]:
        path = os.path.join(folder, label_name)
        label = 0 if label_name == "cats" else 1
        if not os.path.exists(path):
            print(f"Folder not found: {path}")
            continue
        files = sorted([f for f in os.listdir(path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        for f in tqdm(files, desc=f"Loading {label_name} images"):
            img_path = os.path.join(path, f)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img = cv2.resize(img, (image_size, image_size))
            images.append(img)
            labels.append(label)
    return np.array(images), np.array(labels)

# -- Step 2 & 3: Extract HOG + ORB in parallel

def extract_features_parallel(images, orb_features=200, image_size=32):
    orb = cv2.ORB_create(nfeatures=orb_features)

    def process(img):
        fd = hog(img, orientations=9, pixels_per_cell=(8,8),
                 cells_per_block=(2,2), block_norm='L2-Hys')
        keypoints, des = orb.detectAndCompute(img, None)
        if des is None:
            des = np.array([], dtype=np.float32).reshape(0, 32)
        return fd, des

    with ThreadPoolExecutor(max_workers=8) as executor:
        results = list(executor.map(process, images))

    hog_features = [r[0] for r in results]
    img_descriptors = [r[1] for r in results]
    all_descriptors = [r[1] for r in results if r[1].shape[0] > 0]

    if all_descriptors:
        all_descriptors = np.vstack(all_descriptors)
    else:
        all_descriptors = np.array([], dtype=np.float32).reshape(0, 32)

    return np.array(hog_features), img_descriptors, all_descriptors

# -- Step 4: Create visual vocabulary (codebook) using fewer clusters for speed

def create_codebook(descriptors, k=30):
    print("Clustering descriptors with MiniBatchKMeans...")
    kmeans = MiniBatchKMeans(n_clusters=k, batch_size=500, random_state=42)
    kmeans.fit(descriptors)
    return kmeans

# -- Step 5: Compute BoW histograms

def compute_bow_histograms(kmeans, img_descriptors, k=30):
    histograms = []
    for des in tqdm(img_descriptors, desc="Computing BoW histograms"):
        if des.shape[0] == 0:
            hist = np.zeros(k)
        else:
            labels = kmeans.predict(des)
            hist, _ = np.histogram(labels, bins=np.arange(k+1))
        histograms.append(hist)
    return np.array(histograms)

# -- Load train and test images
X_train_imgs, y_train = load_images("data/train", image_size=32)
X_test_imgs, y_test = load_images("data/test", image_size=32)

# Extract features in parallel
X_train_hog, train_img_des, train_all_des = extract_features_parallel(X_train_imgs, orb_features=200)
X_test_hog, test_img_des, test_all_des = extract_features_parallel(X_test_imgs, orb_features=200)

# Create visual codebook (BoW)
k = 30
if train_all_des.shape[0] == 0:
    print("No ORB descriptors found in training images. Skipping ORB features.")
    X_train_orb_bow = np.zeros((len(X_train_imgs), k))
    X_test_orb_bow = np.zeros((len(X_test_imgs), k))
else:
    kmeans = create_codebook(train_all_des, k=k)
    X_train_orb_bow = compute_bow_histograms(kmeans, train_img_des, k=k)
    X_test_orb_bow = compute_bow_histograms(kmeans, test_img_des, k=k)

# Combine features
X_train_feat = np.hstack([X_train_hog, X_train_orb_bow])
X_test_feat = np.hstack([X_test_hog, X_test_orb_bow])

# Normalize
scaler = StandardScaler()
X_train_feat = scaler.fit_transform(X_train_feat)
X_test_feat = scaler.transform(X_test_feat)

# Shuffle
X_train_feat, y_train = shuffle(X_train_feat, y_train, random_state=42)
X_test_feat, y_test = shuffle(X_test_feat, y_test, random_state=42)

# Train SVM
print("\nTraining LinearSVC with combined HOG + ORB-BoW features...")
clf = LinearSVC(C=1.0, max_iter=5000, dual=False)
clf.fit(X_train_feat, y_train)

# Evaluate
y_pred = clf.predict(X_test_feat)
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=["Cat", "Dog"]))

Loading dogs images: 100%|██████████| 2500/2500 [00:02<00:00, 1177.37it/s]


No ORB descriptors found in training images. Skipping ORB features.

Training LinearSVC with combined HOG + ORB-BoW features...

Accuracy: 0.6984

Classification Report:
               precision    recall  f1-score   support

         Cat       0.71      0.67      0.69      2500
         Dog       0.69      0.73      0.71      2500

    accuracy                           0.70      5000
   macro avg       0.70      0.70      0.70      5000
weighted avg       0.70      0.70      0.70      5000



In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import hog
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import shuffle
from sklearn.cluster import MiniBatchKMeans
from concurrent.futures import ThreadPoolExecutor

# -- Step 1: Load all grayscale images from folder, downsized to 64x64 for better ORB performance

def load_images(folder, image_size=64):
    images = []
    labels = []
    for label_name in ["cats", "dogs"]:
        path = os.path.join(folder, label_name)
        label = 0 if label_name == "cats" else 1
        if not os.path.exists(path):
            print(f"Folder not found: {path}")
            continue
        files = sorted([f for f in os.listdir(path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        for idx, f in enumerate(tqdm(files, desc=f"Loading {label_name} images")):
            img_path = os.path.join(path, f)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img = cv2.resize(img, (image_size, image_size))
            images.append(img)
            labels.append(label)
    return np.array(images), np.array(labels)

# -- Step 2 & 3: Extract HOG + ORB in parallel with debugging and reduced ORB threshold

def extract_features_parallel(images, orb_features=200, image_size=64):
    orb = cv2.ORB_create(nfeatures=orb_features, fastThreshold=5)

    def process(img):
        fd = hog(img, orientations=9, pixels_per_cell=(8,8),
                 cells_per_block=(2,2), block_norm='L2-Hys')
        keypoints, des = orb.detectAndCompute(img, None)
        if des is None or des.shape[0] == 0:
            des = np.array([], dtype=np.float32).reshape(0, 32)
        return fd, des

    with ThreadPoolExecutor(max_workers=8) as executor:
        results = list(executor.map(process, images))

    hog_features = [r[0] for r in results]
    img_descriptors = [r[1] for r in results]
    all_descriptors = [r[1] for r in results if r[1].shape[0] > 0]

    if all_descriptors:
        all_descriptors = np.vstack(all_descriptors)
    else:
        print("Warning: No ORB descriptors found in the dataset. Check image quality or increase size.")
        all_descriptors = np.array([], dtype=np.float32).reshape(0, 32)

    return np.array(hog_features), img_descriptors, all_descriptors

# -- Step 4: Create visual vocabulary (codebook) using fewer clusters for speed

def create_codebook(descriptors, k=30):
    print("Clustering descriptors with MiniBatchKMeans...")
    kmeans = MiniBatchKMeans(n_clusters=k, batch_size=500, random_state=42)
    kmeans.fit(descriptors)
    return kmeans

# -- Step 5: Compute BoW histograms

def compute_bow_histograms(kmeans, img_descriptors, k=30):
    histograms = []
    for des in tqdm(img_descriptors, desc="Computing BoW histograms"):
        if des.shape[0] == 0:
            hist = np.zeros(k)
        else:
            labels = kmeans.predict(des)
            hist, _ = np.histogram(labels, bins=np.arange(k+1))
        histograms.append(hist)
    return np.array(histograms)

# -- Load train and test images
X_train_imgs, y_train = load_images("data/train", image_size=64)
X_test_imgs, y_test = load_images("data/test", image_size=64)

# Extract features in parallel
X_train_hog, train_img_des, train_all_des = extract_features_parallel(X_train_imgs, orb_features=300)
X_test_hog, test_img_des, test_all_des = extract_features_parallel(X_test_imgs, orb_features=300)

# Create visual codebook (BoW)
k = 50
if train_all_des.shape[0] == 0:
    print("No ORB descriptors found in training images. Skipping ORB features.")
    X_train_orb_bow = np.zeros((len(X_train_imgs), k))
    X_test_orb_bow = np.zeros((len(X_test_imgs), k))
else:
    kmeans = create_codebook(train_all_des, k=k)
    X_train_orb_bow = compute_bow_histograms(kmeans, train_img_des, k=k)
    X_test_orb_bow = compute_bow_histograms(kmeans, test_img_des, k=k)

# Combine features
X_train_feat = np.hstack([X_train_hog, X_train_orb_bow])
X_test_feat = np.hstack([X_test_hog, X_test_orb_bow])

# Normalize
scaler = StandardScaler()
X_train_feat = scaler.fit_transform(X_train_feat)
X_test_feat = scaler.transform(X_test_feat)

# Shuffle
X_train_feat, y_train = shuffle(X_train_feat, y_train, random_state=42)
X_test_feat, y_test = shuffle(X_test_feat, y_test, random_state=42)

# Train SVM
print("\nTraining LinearSVC with combined HOG + ORB-BoW features...")
clf = LinearSVC(C=1.0, max_iter=5000, dual=False)
clf.fit(X_train_feat, y_train)

# Evaluate
y_pred = clf.predict(X_test_feat)
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=["Cat", "Dog"]))

Loading dogs images: 100%|██████████| 2500/2500 [00:01<00:00, 1437.40it/s]


Clustering descriptors with MiniBatchKMeans...


Computing BoW histograms: 100%|██████████| 5000/5000 [00:00<00:00, 10045.87it/s]



Training LinearSVC with combined HOG + ORB-BoW features...

Accuracy: 0.7164

Classification Report:
               precision    recall  f1-score   support

         Cat       0.72      0.70      0.71      2500
         Dog       0.71      0.73      0.72      2500

    accuracy                           0.72      5000
   macro avg       0.72      0.72      0.72      5000
weighted avg       0.72      0.72      0.72      5000



parameter tuning

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import hog
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import shuffle
from sklearn.cluster import MiniBatchKMeans
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import StratifiedKFold

import random

# Load images function (unchanged)
def load_images(folder, image_size=64):
    images = []
    labels = []
    for label_name in ["cats", "dogs"]:
        path = os.path.join(folder, label_name)
        label = 0 if label_name == "cats" else 1
        if not os.path.exists(path):
            print(f"Folder not found: {path}")
            continue
        files = sorted([f for f in os.listdir(path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        for idx, f in enumerate(tqdm(files, desc=f"Loading {label_name} images")):
            img_path = os.path.join(path, f)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img = cv2.resize(img, (image_size, image_size))
            images.append(img)
            labels.append(label)
    return np.array(images), np.array(labels)

# Feature extraction with params orb_features (number of ORB keypoints)
def extract_features_parallel(images, orb_features=200, image_size=64):
    orb = cv2.ORB_create(nfeatures=orb_features, fastThreshold=5)

    def process(img):
        fd = hog(img, orientations=9, pixels_per_cell=(8,8),
                 cells_per_block=(2,2), block_norm='L2-Hys')
        keypoints, des = orb.detectAndCompute(img, None)
        if des is None or des.shape[0] == 0:
            des = np.array([], dtype=np.float32).reshape(0, 32)
        return fd, des

    with ThreadPoolExecutor(max_workers=8) as executor:
        results = list(executor.map(process, images))

    hog_features = [r[0] for r in results]
    img_descriptors = [r[1] for r in results]
    all_descriptors = [r[1] for r in results if r[1].shape[0] > 0]

    if all_descriptors:
        all_descriptors = np.vstack(all_descriptors)
    else:
        print("Warning: No ORB descriptors found in the dataset. Check image quality or increase size.")
        all_descriptors = np.array([], dtype=np.float32).reshape(0, 32)

    return np.array(hog_features), img_descriptors, all_descriptors

# Create codebook (k clusters)
def create_codebook(descriptors, k=30):
    kmeans = MiniBatchKMeans(n_clusters=k, batch_size=500, random_state=42)
    kmeans.fit(descriptors)
    return kmeans

# Compute BoW histograms from descriptors
def compute_bow_histograms(kmeans, img_descriptors, k=30):
    histograms = []
    for des in img_descriptors:
        if des.shape[0] == 0:
            hist = np.zeros(k)
        else:
            labels = kmeans.predict(des)
            hist, _ = np.histogram(labels, bins=np.arange(k+1))
        histograms.append(hist)
    return np.array(histograms)

# Function to run full pipeline with given params and return validation accuracy
def run_pipeline(X, y, k, orb_features, C, max_iter, n_splits=3):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    accuracies = []
    for train_idx, val_idx in skf.split(X, y):
        X_train_imgs, X_val_imgs = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # Feature extraction
        X_train_hog, train_img_des, train_all_des = extract_features_parallel(X_train_imgs, orb_features=orb_features)
        X_val_hog, val_img_des, val_all_des = extract_features_parallel(X_val_imgs, orb_features=orb_features)

        if train_all_des.shape[0] == 0:
            # No ORB features found: fallback to HOG only
            X_train_orb_bow = np.zeros((len(X_train_imgs), k))
            X_val_orb_bow = np.zeros((len(X_val_imgs), k))
        else:
            kmeans = create_codebook(train_all_des, k=k)
            X_train_orb_bow = compute_bow_histograms(kmeans, train_img_des, k=k)
            X_val_orb_bow = compute_bow_histograms(kmeans, val_img_des, k=k)

        X_train_feat = np.hstack([X_train_hog, X_train_orb_bow])
        X_val_feat = np.hstack([X_val_hog, X_val_orb_bow])

        scaler = StandardScaler()
        X_train_feat = scaler.fit_transform(X_train_feat)
        X_val_feat = scaler.transform(X_val_feat)

        clf = LinearSVC(C=C, max_iter=max_iter, dual=False)
        clf.fit(X_train_feat, y_train)
        y_pred = clf.predict(X_val_feat)
        acc = accuracy_score(y_val, y_pred)
        accuracies.append(acc)

    mean_acc = np.mean(accuracies)
    return mean_acc

# Load all images once
X_all_imgs, y_all = load_images("data/train", image_size=64)

# Define search space
param_space = {
    "k": [20, 30, 40],  # clusters in codebook
    "orb_features": [100, 200, 300],  # ORB features
    "C": [0.1, 1.0, 10.0],
    "max_iter": [1000, 3000, 5000]
}

# Manual random search
n_iter_search = 6
best_score = 0
best_params = None

for i in range(n_iter_search):
    params = {
        "k": random.choice(param_space["k"]),
        "orb_features": random.choice(param_space["orb_features"]),
        "C": random.choice(param_space["C"]),
        "max_iter": random.choice(param_space["max_iter"]),
    }
    print(f"\nIteration {i+1}/{n_iter_search} with params: {params}")
    score = run_pipeline(X_all_imgs, y_all,
                         k=params["k"],
                         orb_features=params["orb_features"],
                         C=params["C"],
                         max_iter=params["max_iter"],
                         n_splits=3)
    print(f"Mean CV accuracy: {score:.4f}")
    if score > best_score:
        best_score = score
        best_params = params

print(f"\nBest params found: {best_params}")
print(f"Best CV accuracy: {best_score:.4f}")

# After tuning, you can train final model on full training data with best params and test on test set...



Loading dogs images: 100%|██████████| 9989/9989 [00:06<00:00, 1434.10it/s]



Iteration 1/6 with params: {'k': 20, 'orb_features': 300, 'C': 10.0, 'max_iter': 3000}
Mean CV accuracy: 0.7014

Iteration 2/6 with params: {'k': 20, 'orb_features': 100, 'C': 10.0, 'max_iter': 1000}
Mean CV accuracy: 0.7014

Iteration 3/6 with params: {'k': 40, 'orb_features': 100, 'C': 0.1, 'max_iter': 5000}
Mean CV accuracy: 0.7013

Iteration 4/6 with params: {'k': 30, 'orb_features': 300, 'C': 0.1, 'max_iter': 1000}
Mean CV accuracy: 0.7021

Iteration 5/6 with params: {'k': 30, 'orb_features': 300, 'C': 10.0, 'max_iter': 1000}
Mean CV accuracy: 0.7010

Iteration 6/6 with params: {'k': 20, 'orb_features': 300, 'C': 1.0, 'max_iter': 1000}
Mean CV accuracy: 0.7014

Best params found: {'k': 30, 'orb_features': 300, 'C': 0.1, 'max_iter': 1000}
Best CV accuracy: 0.7021


In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import hog
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import shuffle
from sklearn.cluster import MiniBatchKMeans
from concurrent.futures import ThreadPoolExecutor

# -- Step 1: Load all grayscale images from folder, downsized to 64x64 for better ORB performance

def load_images(folder, image_size=64):
    images = []
    labels = []
    for label_name in ["cats", "dogs"]:
        path = os.path.join(folder, label_name)
        label = 0 if label_name == "cats" else 1
        if not os.path.exists(path):
            print(f"Folder not found: {path}")
            continue
        files = sorted([f for f in os.listdir(path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        for idx, f in enumerate(tqdm(files, desc=f"Loading {label_name} images")):
            img_path = os.path.join(path, f)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img = cv2.resize(img, (image_size, image_size))
            images.append(img)
            labels.append(label)
    return np.array(images), np.array(labels)

# -- Step 2 & 3: Extract HOG + ORB in parallel with debugging and reduced ORB threshold

def extract_features_parallel(images, orb_features=300, image_size=64):
    orb = cv2.ORB_create(nfeatures=orb_features, fastThreshold=5)

    def process(img):
        fd = hog(img, orientations=9, pixels_per_cell=(8,8),
                 cells_per_block=(2,2), block_norm='L2-Hys')
        keypoints, des = orb.detectAndCompute(img, None)
        if des is None or des.shape[0] == 0:
            des = np.array([], dtype=np.float32).reshape(0, 32)
        return fd, des

    with ThreadPoolExecutor(max_workers=8) as executor:
        results = list(executor.map(process, images))

    hog_features = [r[0] for r in results]
    img_descriptors = [r[1] for r in results]
    all_descriptors = [r[1] for r in results if r[1].shape[0] > 0]

    if all_descriptors:
        all_descriptors = np.vstack(all_descriptors)
    else:
        print("Warning: No ORB descriptors found in the dataset. Check image quality or increase size.")
        all_descriptors = np.array([], dtype=np.float32).reshape(0, 32)

    return np.array(hog_features), img_descriptors, all_descriptors

# -- Step 4: Create visual vocabulary (codebook) using k=30 (best from tuning)

def create_codebook(descriptors, k=30):
    print("Clustering descriptors with MiniBatchKMeans...")
    kmeans = MiniBatchKMeans(n_clusters=k, batch_size=500, random_state=42)
    kmeans.fit(descriptors)
    return kmeans

# -- Step 5: Compute BoW histograms

def compute_bow_histograms(kmeans, img_descriptors, k=30):
    histograms = []
    for des in tqdm(img_descriptors, desc="Computing BoW histograms"):
        if des.shape[0] == 0:
            hist = np.zeros(k)
        else:
            labels = kmeans.predict(des)
            hist, _ = np.histogram(labels, bins=np.arange(k+1))
        histograms.append(hist)
    return np.array(histograms)

# -- Load train and test images
X_train_imgs, y_train = load_images("data/train", image_size=64)
X_test_imgs, y_test = load_images("data/test", image_size=64)

# Extract features in parallel using best parameters
global_k = 30
global_orb_features = 300
X_train_hog, train_img_des, train_all_des = extract_features_parallel(X_train_imgs, orb_features=global_orb_features)
X_test_hog, test_img_des, test_all_des = extract_features_parallel(X_test_imgs, orb_features=global_orb_features)

# Create visual codebook (BoW)
if train_all_des.shape[0] == 0:
    print("No ORB descriptors found in training images. Skipping ORB features.")
    X_train_orb_bow = np.zeros((len(X_train_imgs), global_k))
    X_test_orb_bow = np.zeros((len(X_test_imgs), global_k))
else:
    kmeans = create_codebook(train_all_des, k=global_k)
    X_train_orb_bow = compute_bow_histograms(kmeans, train_img_des, k=global_k)
    X_test_orb_bow = compute_bow_histograms(kmeans, test_img_des, k=global_k)

# Combine features
X_train_feat = np.hstack([X_train_hog, X_train_orb_bow])
X_test_feat = np.hstack([X_test_hog, X_test_orb_bow])

# Normalize
scaler = StandardScaler()
X_train_feat = scaler.fit_transform(X_train_feat)
X_test_feat = scaler.transform(X_test_feat)

# Shuffle
X_train_feat, y_train = shuffle(X_train_feat, y_train, random_state=42)
X_test_feat, y_test = shuffle(X_test_feat, y_test, random_state=42)

# Train SVM with best hyperparameters from search
print("\nTraining LinearSVC with tuned hyperparameters...")
clf = LinearSVC(C=0.1, max_iter=1000, dual=False)
clf.fit(X_train_feat, y_train)

# Evaluate
y_pred = clf.predict(X_test_feat)
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=["Cat", "Dog"]))


Loading dogs images: 100%|██████████| 2500/2500 [00:01<00:00, 1409.33it/s]


Clustering descriptors with MiniBatchKMeans...


Computing BoW histograms: 100%|██████████| 5000/5000 [00:00<00:00, 6730.11it/s]



Training LinearSVC with tuned hyperparameters...

Accuracy: 0.7148

Classification Report:
               precision    recall  f1-score   support

         Cat       0.72      0.70      0.71      2500
         Dog       0.71      0.73      0.72      2500

    accuracy                           0.71      5000
   macro avg       0.71      0.71      0.71      5000
weighted avg       0.71      0.71      0.71      5000



with preprocessor

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import hog
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import shuffle
from sklearn.cluster import MiniBatchKMeans
from concurrent.futures import ThreadPoolExecutor

# Preprocessor class to modularize feature extraction pipeline
class SVMPreprocessor:
    def __init__(self, image_size=64, orb_features=300, k=30):
        self.image_size = image_size
        self.orb_features = orb_features
        self.k = k
        self.scaler = StandardScaler()
        self.kmeans = None

    def load_images(self, folder):
        images, labels = [], []
        for label_name in ["cats", "dogs"]:
            path = os.path.join(folder, label_name)
            label = 0 if label_name == "cats" else 1
            if not os.path.exists(path):
                print(f"Folder not found: {path}")
                continue
            files = sorted([f for f in os.listdir(path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            for f in tqdm(files, desc=f"Loading {label_name} images"):
                img_path = os.path.join(path, f)
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    continue
                img = cv2.resize(img, (self.image_size, self.image_size))
                images.append(img)
                labels.append(label)
        return np.array(images), np.array(labels)

    def extract_features_parallel(self, images):
        orb = cv2.ORB_create(nfeatures=self.orb_features, fastThreshold=5)

        def process(img):
            fd = hog(img, orientations=9, pixels_per_cell=(8,8),
                     cells_per_block=(2,2), block_norm='L2-Hys')
            keypoints, des = orb.detectAndCompute(img, None)
            if des is None or des.shape[0] == 0:
                des = np.array([], dtype=np.float32).reshape(0, 32)
            return fd, des

        with ThreadPoolExecutor(max_workers=8) as executor:
            results = list(executor.map(process, images))

        hog_features = [r[0] for r in results]
        img_descriptors = [r[1] for r in results]
        all_descriptors = [r[1] for r in results if r[1].shape[0] > 0]

        if all_descriptors:
            all_descriptors = np.vstack(all_descriptors)
        else:
            print("Warning: No ORB descriptors found in the dataset.")
            all_descriptors = np.array([], dtype=np.float32).reshape(0, 32)

        return np.array(hog_features), img_descriptors, all_descriptors

    def create_codebook(self, descriptors):
        print("Clustering descriptors with MiniBatchKMeans...")
        self.kmeans = MiniBatchKMeans(n_clusters=self.k, batch_size=500, random_state=42)
        self.kmeans.fit(descriptors)

    def compute_bow_histograms(self, img_descriptors):
        histograms = []
        for des in tqdm(img_descriptors, desc="Computing BoW histograms"):
            if des.shape[0] == 0:
                hist = np.zeros(self.k)
            else:
                labels = self.kmeans.predict(des)
                hist, _ = np.histogram(labels, bins=np.arange(self.k+1))
            histograms.append(hist)
        return np.array(histograms)

    def transform_images(self, images):
        hog_feats, img_des, all_des = self.extract_features_parallel(images)
        if self.kmeans is None:
            self.create_codebook(all_des)
        orb_bow = self.compute_bow_histograms(img_des)
        combined = np.hstack([hog_feats, orb_bow])
        return self.scaler.transform(combined)

    def fit_transform_images(self, images):
        hog_feats, img_des, all_des = self.extract_features_parallel(images)
        self.create_codebook(all_des)
        orb_bow = self.compute_bow_histograms(img_des)
        combined = np.hstack([hog_feats, orb_bow])
        return self.scaler.fit_transform(combined)

# Instantiate and run pipeline
preprocessor = SVMPreprocessor(image_size=64, orb_features=300, k=30)
X_train_imgs, y_train = preprocessor.load_images("data/train")
X_test_imgs, y_test = preprocessor.load_images("data/test")

X_train_feat = preprocessor.fit_transform_images(X_train_imgs)
X_test_feat = preprocessor.transform_images(X_test_imgs)

X_train_feat, y_train = shuffle(X_train_feat, y_train, random_state=42)
X_test_feat, y_test = shuffle(X_test_feat, y_test, random_state=42)

print("\nTraining LinearSVC with tuned hyperparameters...")
clf = LinearSVC(C=0.1, max_iter=1000, dual=False)
clf.fit(X_train_feat, y_train)

y_pred = clf.predict(X_test_feat)
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=["Cat", "Dog"]))

Loading dogs images: 100%|██████████| 2500/2500 [00:01<00:00, 1430.85it/s]


Clustering descriptors with MiniBatchKMeans...


Computing BoW histograms: 100%|██████████| 5000/5000 [00:00<00:00, 9983.86it/s] 



Training LinearSVC with tuned hyperparameters...

Accuracy: 0.7148

Classification Report:
               precision    recall  f1-score   support

         Cat       0.72      0.70      0.71      2500
         Dog       0.71      0.73      0.72      2500

    accuracy                           0.71      5000
   macro avg       0.71      0.71      0.71      5000
weighted avg       0.71      0.71      0.71      5000



some feature engineering

In [1]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import hog
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import shuffle
from sklearn.cluster import MiniBatchKMeans
from concurrent.futures import ThreadPoolExecutor

# Preprocessor class to modularize feature extraction pipeline
class SVMPreprocessor:
    def __init__(self, image_size=64, orb_features=300, k=30):
        self.image_size = image_size
        self.orb_features = orb_features
        self.k = k
        self.scaler = StandardScaler()
        self.kmeans = None

    def load_images(self, folder):
        images, labels = [], []
        for label_name in ["cats", "dogs"]:
            path = os.path.join(folder, label_name)
            label = 0 if label_name == "cats" else 1
            if not os.path.exists(path):
                print(f"Folder not found: {path}")
                continue
            files = sorted([f for f in os.listdir(path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            for f in tqdm(files, desc=f"Loading {label_name} images"):
                img_path = os.path.join(path, f)
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    continue
                img = cv2.resize(img, (self.image_size, self.image_size))
                images.append(img)
                labels.append(label)
        return np.array(images), np.array(labels)

    def extract_features_parallel(self, images):
        orb = cv2.ORB_create(nfeatures=self.orb_features, fastThreshold=5)

        def process(img):
            fd = hog(img, orientations=9, pixels_per_cell=(8,8),
                     cells_per_block=(2,2), block_norm='L2-Hys')
            keypoints, des = orb.detectAndCompute(img, None)
            if des is None or des.shape[0] == 0:
                des = np.array([], dtype=np.float32).reshape(0, 32)

            # Feature engineering: mean intensity and edge density
            mean_intensity = np.mean(img) / 255.0
            edges = cv2.Canny(img, 100, 200)
            edge_density = np.sum(edges > 0) / (self.image_size ** 2)

            return fd, des, mean_intensity, edge_density

        with ThreadPoolExecutor(max_workers=8) as executor:
            results = list(executor.map(process, images))

        hog_features = [r[0] for r in results]
        img_descriptors = [r[1] for r in results]
        all_descriptors = [r[1] for r in results if r[1].shape[0] > 0]
        engineered_feats = [[r[2], r[3]] for r in results]

        if all_descriptors:
            all_descriptors = np.vstack(all_descriptors)
        else:
            print("Warning: No ORB descriptors found in the dataset.")
            all_descriptors = np.array([], dtype=np.float32).reshape(0, 32)

        return np.array(hog_features), img_descriptors, all_descriptors, np.array(engineered_feats)

    def create_codebook(self, descriptors):
        print("Clustering descriptors with MiniBatchKMeans...")
        self.kmeans = MiniBatchKMeans(n_clusters=self.k, batch_size=500, random_state=42)
        self.kmeans.fit(descriptors)

    def compute_bow_histograms(self, img_descriptors):
        histograms = []
        for des in tqdm(img_descriptors, desc="Computing BoW histograms"):
            if des.shape[0] == 0:
                hist = np.zeros(self.k)
            else:
                labels = self.kmeans.predict(des)
                hist, _ = np.histogram(labels, bins=np.arange(self.k+1))
            histograms.append(hist)
        return np.array(histograms)

    def transform_images(self, images):
        hog_feats, img_des, _, engineered_feats = self.extract_features_parallel(images)
        orb_bow = self.compute_bow_histograms(img_des)
        combined = np.hstack([hog_feats, orb_bow, engineered_feats])
        return self.scaler.transform(combined)

    def fit_transform_images(self, images):
        hog_feats, img_des, all_des, engineered_feats = self.extract_features_parallel(images)
        self.create_codebook(all_des)
        orb_bow = self.compute_bow_histograms(img_des)
        combined = np.hstack([hog_feats, orb_bow, engineered_feats])
        return self.scaler.fit_transform(combined)

# Instantiate and run pipeline
preprocessor = SVMPreprocessor(image_size=64, orb_features=300, k=30)
X_train_imgs, y_train = preprocessor.load_images("data/train")
X_test_imgs, y_test = preprocessor.load_images("data/test")

X_train_feat = preprocessor.fit_transform_images(X_train_imgs)
X_test_feat = preprocessor.transform_images(X_test_imgs)

X_train_feat, y_train = shuffle(X_train_feat, y_train, random_state=42)
X_test_feat, y_test = shuffle(X_test_feat, y_test, random_state=42)

print("\nTraining LinearSVC with tuned hyperparameters...")
clf = LinearSVC(C=0.1, max_iter=1000, dual=False)
clf.fit(X_train_feat, y_train)

y_pred = clf.predict(X_test_feat)
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=["Cat", "Dog"]))

Folder not found: data/train/cats
Folder not found: data/train/dogs
Folder not found: data/test/cats
Folder not found: data/test/dogs
Clustering descriptors with MiniBatchKMeans...


ValueError: Found array with 0 sample(s) (shape=(0, 32)) while a minimum of 1 is required by MiniBatchKMeans.

a try with rbf

In [2]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import hog, local_binary_pattern
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import shuffle
from sklearn.cluster import MiniBatchKMeans
from concurrent.futures import ThreadPoolExecutor

# Preprocessor class to modularize feature extraction pipeline
class SVMPreprocessor:
    def __init__(self, image_size=64, orb_features=300, k=30):
        self.image_size = image_size
        self.orb_features = orb_features
        self.k = k
        self.scaler = StandardScaler()
        self.kmeans = None

    def load_images(self, folder):
        images, labels = [], []
        for label_name in ["cats", "dogs"]:
            path = os.path.join(folder, label_name)
            label = 0 if label_name == "cats" else 1
            if not os.path.exists(path):
                print(f"Folder not found: {path}")
                continue
            files = sorted([f for f in os.listdir(path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            for f in tqdm(files, desc=f"Loading {label_name} images"):
                img_path = os.path.join(path, f)
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    continue
                img = cv2.resize(img, (self.image_size, self.image_size))
                images.append(img)
                labels.append(label)
        return np.array(images), np.array(labels)

    def extract_features_parallel(self, images):
        orb = cv2.ORB_create(nfeatures=self.orb_features, fastThreshold=5)

        def process(img):
            fd = hog(img, orientations=9, pixels_per_cell=(8,8),
                     cells_per_block=(2,2), block_norm='L2-Hys')
            keypoints, des = orb.detectAndCompute(img, None)
            if des is None or des.shape[0] == 0:
                des = np.array([], dtype=np.float32).reshape(0, 32)

            # Feature engineering: mean intensity and edge density
            mean_intensity = np.mean(img) / 255.0
            edges = cv2.Canny(img, 100, 200)
            edge_density = np.sum(edges > 0) / (self.image_size ** 2)

            # Feature engineering: LBP histogram
            lbp = local_binary_pattern(img, P=8, R=1, method="uniform")
            (hist, _) = np.histogram(lbp.ravel(), bins=np.arange(0, 11), range=(0, 10))
            hist = hist.astype("float")
            hist /= (hist.sum() + 1e-7)

            return fd, des, mean_intensity, edge_density, hist

        with ThreadPoolExecutor(max_workers=8) as executor:
            results = list(executor.map(process, images))

        hog_features = [r[0] for r in results]
        img_descriptors = [r[1] for r in results]
        all_descriptors = [r[1] for r in results if r[1].shape[0] > 0]
        engineered_feats = [[r[2], r[3]] for r in results]
        lbp_feats = [r[4] for r in results]

        if all_descriptors:
            all_descriptors = np.vstack(all_descriptors)
        else:
            print("Warning: No ORB descriptors found in the dataset.")
            all_descriptors = np.array([], dtype=np.float32).reshape(0, 32)

        return np.array(hog_features), img_descriptors, all_descriptors, np.array(engineered_feats), np.array(lbp_feats)

    def create_codebook(self, descriptors):
        print("Clustering descriptors with MiniBatchKMeans...")
        self.kmeans = MiniBatchKMeans(n_clusters=self.k, batch_size=500, random_state=42)
        self.kmeans.fit(descriptors)

    def compute_bow_histograms(self, img_descriptors):
        histograms = []
        for des in tqdm(img_descriptors, desc="Computing BoW histograms"):
            if des.shape[0] == 0:
                hist = np.zeros(self.k)
            else:
                labels = self.kmeans.predict(des)
                hist, _ = np.histogram(labels, bins=np.arange(self.k+1))
            histograms.append(hist)
        return np.array(histograms)

    def transform_images(self, images):
        hog_feats, img_des, _, engineered_feats, lbp_feats = self.extract_features_parallel(images)
        orb_bow = self.compute_bow_histograms(img_des)
        combined = np.hstack([hog_feats, orb_bow, engineered_feats, lbp_feats])
        return self.scaler.transform(combined)

    def fit_transform_images(self, images):
        hog_feats, img_des, all_des, engineered_feats, lbp_feats = self.extract_features_parallel(images)
        self.create_codebook(all_des)
        orb_bow = self.compute_bow_histograms(img_des)
        combined = np.hstack([hog_feats, orb_bow, engineered_feats, lbp_feats])
        return self.scaler.fit_transform(combined)

# Instantiate and run pipeline
preprocessor = SVMPreprocessor(image_size=64, orb_features=300, k=30)
X_train_imgs, y_train = preprocessor.load_images("/content/dog-cat-full-dataset/data/train")
X_test_imgs, y_test = preprocessor.load_images("/content/dog-cat-full-dataset/data/test")

X_train_feat = preprocessor.fit_transform_images(X_train_imgs)
X_test_feat = preprocessor.transform_images(X_test_imgs)

X_train_feat, y_train = shuffle(X_train_feat, y_train, random_state=42)
X_test_feat, y_test = shuffle(X_test_feat, y_test, random_state=42)

print("\nTraining RBF SVC with tuned hyperparameters...")
clf = SVC(C=1.0, kernel="rbf", gamma='scale', max_iter=10000)
clf.fit(X_train_feat, y_train)

y_pred = clf.predict(X_test_feat)
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=["Cat", "Dog"]))


Loading dogs images: 100%|██████████| 2500/2500 [00:02<00:00, 989.53it/s] 


Clustering descriptors with MiniBatchKMeans...


Computing BoW histograms: 100%|██████████| 5000/5000 [00:00<00:00, 8225.72it/s]



Training RBF SVC with tuned hyperparameters...


/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(



Accuracy: 0.7782

Classification Report:
               precision    recall  f1-score   support

         Cat       0.78      0.78      0.78      2500
         Dog       0.78      0.78      0.78      2500

    accuracy                           0.78      5000
   macro avg       0.78      0.78      0.78      5000
weighted avg       0.78      0.78      0.78      5000



Principal Component Analysis

In [3]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import hog, local_binary_pattern
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import shuffle
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from concurrent.futures import ThreadPoolExecutor

# Preprocessor class to modularize feature extraction pipeline
class SVMPreprocessor:
    def __init__(self, image_size=64, orb_features=300, k=30):
        self.image_size = image_size
        self.orb_features = orb_features
        self.k = k
        self.scaler = StandardScaler()
        self.kmeans = None

    def load_images(self, folder):
        images, labels = [], []
        for label_name in ["cats", "dogs"]:
            path = os.path.join(folder, label_name)
            label = 0 if label_name == "cats" else 1
            if not os.path.exists(path):
                print(f"Folder not found: {path}")
                continue
            files = sorted([f for f in os.listdir(path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            for f in tqdm(files, desc=f"Loading {label_name} images"):
                img_path = os.path.join(path, f)
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    continue
                img = cv2.resize(img, (self.image_size, self.image_size))
                images.append(img)
                labels.append(label)
        return np.array(images), np.array(labels)

    def extract_features_parallel(self, images):
        orb = cv2.ORB_create(nfeatures=self.orb_features, fastThreshold=5)

        def process(img):
            fd = hog(img, orientations=9, pixels_per_cell=(8,8),
                     cells_per_block=(2,2), block_norm='L2-Hys')
            keypoints, des = orb.detectAndCompute(img, None)
            if des is None or des.shape[0] == 0:
                des = np.array([], dtype=np.float32).reshape(0, 32)

            # Feature engineering: mean intensity and edge density
            mean_intensity = np.mean(img) / 255.0
            edges = cv2.Canny(img, 100, 200)
            edge_density = np.sum(edges > 0) / (self.image_size ** 2)

            # Feature engineering: LBP histogram
            lbp = local_binary_pattern(img, P=8, R=1, method="uniform")
            (hist, _) = np.histogram(lbp.ravel(), bins=np.arange(0, 11), range=(0, 10))
            hist = hist.astype("float")
            hist /= (hist.sum() + 1e-7)

            return fd, des, mean_intensity, edge_density, hist

        with ThreadPoolExecutor(max_workers=8) as executor:
            results = list(executor.map(process, images))

        hog_features = [r[0] for r in results]
        img_descriptors = [r[1] for r in results]
        all_descriptors = [r[1] for r in results if r[1].shape[0] > 0]
        engineered_feats = [[r[2], r[3]] for r in results]
        lbp_feats = [r[4] for r in results]

        if all_descriptors:
            all_descriptors = np.vstack(all_descriptors)
        else:
            print("Warning: No ORB descriptors found in the dataset.")
            all_descriptors = np.array([], dtype=np.float32).reshape(0, 32)

        return np.array(hog_features), img_descriptors, all_descriptors, np.array(engineered_feats), np.array(lbp_feats)

    def create_codebook(self, descriptors):
        print("Clustering descriptors with MiniBatchKMeans...")
        self.kmeans = MiniBatchKMeans(n_clusters=self.k, batch_size=500, random_state=42)
        self.kmeans.fit(descriptors)

    def compute_bow_histograms(self, img_descriptors):
        histograms = []
        for des in tqdm(img_descriptors, desc="Computing BoW histograms"):
            if des.shape[0] == 0:
                hist = np.zeros(self.k)
            else:
                labels = self.kmeans.predict(des)
                hist, _ = np.histogram(labels, bins=np.arange(self.k+1))
            histograms.append(hist)
        return np.array(histograms)

    def transform_images(self, images):
        hog_feats, img_des, _, engineered_feats, lbp_feats = self.extract_features_parallel(images)
        orb_bow = self.compute_bow_histograms(img_des)
        combined = np.hstack([hog_feats, orb_bow, engineered_feats, lbp_feats])
        return self.scaler.transform(combined)

    def fit_transform_images(self, images):
        hog_feats, img_des, all_des, engineered_feats, lbp_feats = self.extract_features_parallel(images)
        self.create_codebook(all_des)
        orb_bow = self.compute_bow_histograms(img_des)
        combined = np.hstack([hog_feats, orb_bow, engineered_feats, lbp_feats])
        return self.scaler.fit_transform(combined)

# Instantiate and run pipeline
preprocessor = SVMPreprocessor(image_size=64, orb_features=300, k=30)
X_train_imgs, y_train = preprocessor.load_images("/content/dog-cat-full-dataset/data/train")
X_test_imgs, y_test = preprocessor.load_images("/content/dog-cat-full-dataset/data/test")

X_train_feat = preprocessor.fit_transform_images(X_train_imgs)
X_test_feat = preprocessor.transform_images(X_test_imgs)

X_train_feat, y_train = shuffle(X_train_feat, y_train, random_state=42)
X_test_feat, y_test = shuffle(X_test_feat, y_test, random_state=42)

print("\nTraining RBF SVC with PCA...")
pca = PCA(n_components=0.95, random_state=42)
clf = SVC(C=1.0, kernel="rbf", gamma='scale', max_iter=10000)
pipeline = Pipeline([
    ('pca', pca),
    ('svc', clf)
])

pipeline.fit(X_train_feat, y_train)

y_pred = pipeline.predict(X_test_feat)
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=["Cat", "Dog"]))


Loading dogs images: 100%|██████████| 2500/2500 [00:03<00:00, 741.65it/s]


Clustering descriptors with MiniBatchKMeans...


Computing BoW histograms: 100%|██████████| 5000/5000 [00:00<00:00, 5377.22it/s]



Training RBF SVC with PCA...


/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(



Accuracy: 0.7776

Classification Report:
               precision    recall  f1-score   support

         Cat       0.78      0.78      0.78      2500
         Dog       0.78      0.78      0.78      2500

    accuracy                           0.78      5000
   macro avg       0.78      0.78      0.78      5000
weighted avg       0.78      0.78      0.78      5000



identified pipeline is saved

In [5]:
# === PART 1: Train and Save the Model ===
import joblib

# Train the model as before
preprocessor = SVMPreprocessor(image_size=64, orb_features=300, k=30)
X_train_imgs, y_train = preprocessor.load_images("/content/dog-cat-full-dataset/data/train")
X_test_imgs, y_test = preprocessor.load_images("/content/dog-cat-full-dataset/data/test")

X_train_feat = preprocessor.fit_transform_images(X_train_imgs)
X_test_feat = preprocessor.transform_images(X_test_imgs)

X_train_feat, y_train = shuffle(X_train_feat, y_train, random_state=42)
X_test_feat, y_test = shuffle(X_test_feat, y_test, random_state=42)

print("\nTraining RBF SVC with PCA...")
pca = PCA(n_components=0.95, random_state=42)
clf = SVC(C=1.0, kernel="rbf", gamma='scale', max_iter=10000)
pipeline = Pipeline([
    ('pca', pca),
    ('svc', clf)
])

pipeline.fit(X_train_feat, y_train)

# Save the preprocessor and model
joblib.dump(preprocessor, "preprocessor.pkl")
joblib.dump(pipeline, "svm_model.pkl")

# Optional: Evaluate
y_pred = pipeline.predict(X_test_feat)
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=["Cat", "Dog"]))


Loading dogs images: 100%|██████████| 2500/2500 [00:02<00:00, 1054.83it/s]


Clustering descriptors with MiniBatchKMeans...


Computing BoW histograms: 100%|██████████| 5000/5000 [00:00<00:00, 8579.12it/s]



Training RBF SVC with PCA...


/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(



Accuracy: 0.7782

Classification Report:
               precision    recall  f1-score   support

         Cat       0.78      0.78      0.78      2500
         Dog       0.78      0.78      0.78      2500

    accuracy                           0.78      5000
   macro avg       0.78      0.78      0.78      5000
weighted avg       0.78      0.78      0.78      5000



using saved pipeline

In [8]:
# === PART 2: Load and Predict ===
import cv2
import joblib
import numpy as np

# Load model and preprocessor
preprocessor = joblib.load("preprocessor.pkl")
pipeline = joblib.load("svm_model.pkl")

def predict_image(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Image not found or unreadable: {img_path}")
    img = cv2.resize(img, (preprocessor.image_size, preprocessor.image_size))
    features = preprocessor.transform_images([img])
    pred = pipeline.predict(features)[0]
    return "Dog" if pred == 1 else "Cat"

# Example usage:
img_path = "/content/dog-cat-full-dataset/data/test/cats/cat.1002.jpg"
print("Prediction:", predict_image(img_path))


Computing BoW histograms: 100%|██████████| 1/1 [00:00<00:00, 4025.24it/s]

Prediction: Cat


making model into a module

In [9]:
import cv2
import joblib
import numpy as np

class CatDogPredictor:
    def __init__(self, preprocessor_path="preprocessor.pkl", model_path="svm_model.pkl"):
        self.preprocessor = joblib.load(preprocessor_path)
        self.model = joblib.load(model_path)

    def predict(self, img_path):
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f"Image not found or unreadable: {img_path}")
        img = cv2.resize(img, (self.preprocessor.image_size, self.preprocessor.image_size))
        features = self.preprocessor.transform_images([img])
        pred = self.model.predict(features)[0]
        return "Dog" if pred == 1 else "Cat"




Computing BoW histograms: 100%|██████████| 1/1 [00:00<00:00, 4755.45it/s]

Prediction: Cat


In [30]:
from catdogclassify import CatDogPredictor
predictor = CatDogPredictor()
img_path = "/content/downloaded_image.jpg"
print("Prediction:", predictor.predict(img_path))

Computing BoW histograms: 100%|██████████| 1/1 [00:00<00:00, 4798.97it/s]

Prediction: Dog


downloading from outside for test

In [28]:
import requests

def download_image(image_url, save_path="downloaded_image.jpg"):
    response = requests.get(image_url)
    if response.status_code == 200:
        with open(save_path, "wb") as f:
            f.write(response.content)
        print(f"Image downloaded and saved to {save_path}")
    else:
        print(f"Failed to download image. Status code: {response.status_code}")

# Example usage:
image_url = "https://hips.hearstapps.com/hmg-prod/images/small-fluffy-dog-breeds-maltipoo-66300ad363389.jpg?crop=0.8872133207665724xw:1xh;center,top&resize=1200:*"
download_image(image_url)


Image downloaded and saved to downloaded_image.jpg
